In [1]:
import random
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import MNIST

class SyntheticMNISTDataset(Dataset):
    def __init__(self, root="./data", train=True, canvas_size=(256, 256), max_digits=4):
        self.mnist = MNIST(root=root, train=train, download=True)
        self.canvas_size = canvas_size
        self.max_digits = max_digits
        self.digit_size = 28

    def __len__(self):
        return 10000 if self.mnist.train else 2000

    def _has_overlap(self, bbox, bboxes, threshold=0.1):
        if not bboxes:
            return False
        x1, y1, x2, y2 = bbox
        for bx1, by1, bx2, by2 in bboxes:
            ix1, iy1 = max(x1, bx1), max(y1, by1)
            ix2, iy2 = min(x2, bx2), min(y2, by2)
            iw, ih = max(0, ix2 - ix1), max(0, iy2 - iy1)
            intersection = iw * ih
            area1 = (x2 - x1) * (y2 - y1)
            area2 = (bx2 - bx1) * (by2 - by1)
            union = area1 + area2 - intersection
            if intersection / union > threshold:
                return True
        return False

    def __getitem__(self, idx):
        H, W = self.canvas_size
        canvas = np.random.uniform(0.0, 0.15, (H, W)).astype(np.float32)
        
        # Add background clutter lines
        for _ in range(random.randint(2, 5)):
            y = random.randint(0, H - 1)
            canvas[y, :] += np.random.uniform(0.0, 0.1)
            x = random.randint(0, W - 1)
            canvas[:, x] += np.random.uniform(0.0, 0.1)

        boxes, labels = [], []
        num_digits = random.randint(1, self.max_digits)
        attempts = 0
        
        while len(boxes) < num_digits and attempts < 50:
            attempts += 1
            mnist_idx = random.randint(0, len(self.mnist) - 1)
            digit_img, label = self.mnist[mnist_idx]
            digit_np = np.array(digit_img, dtype=np.float32) / 255.0
            
            ymin = random.randint(0, H - self.digit_size)
            xmin = random.randint(0, W - self.digit_size)
            ymax, xmax = ymin + self.digit_size, xmin + self.digit_size
            current_box = [xmin, ymin, xmax, ymax]
            
            if not self._has_overlap(current_box, boxes):
                canvas[ymin:ymax, xmin:xmax] = np.maximum(canvas[ymin:ymax, xmin:xmax], digit_np)
                boxes.append(current_box)
                labels.append(label)
                
        if not boxes:
            return self.__getitem__(idx)

        image_tensor = torch.tensor(canvas, dtype=torch.float32).unsqueeze(0).repeat(3, 1, 1)
        target = {
            "boxes": torch.tensor(boxes, dtype=torch.float32),
            "labels": torch.tensor(labels, dtype=torch.long),
            "image_id": torch.tensor([idx])
        }
        return image_tensor, target

def collate_fn(batch):
    return tuple(zip(*batch))

# --- Instantiation and Verification ---
dataset = SyntheticMNISTDataset(train=True)
data_loader = DataLoader(dataset, batch_size=4, shuffle=True, collate_fn=collate_fn)

# Pull a random sample batch
images, targets = next(iter(data_loader))
print(f"Batch loaded. Image shape: {images[0].shape}")
print(f"Sample targets: {targets[0]}")

c:\Users\student26\Desktop\fasih\venv\Lib\site-packages\torch\_subclasses\functional_tensor.py:377: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:82.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


ModuleNotFoundError: No module named 'numpy'